Starting from Beta(1, 1), process the sequence S,S,F,S,S,F,S. Write the posterior
after each observation and the nal posterior mean. Verify in code.

Note:
Posterior Update:
Beta(alpha, beta) --s success and n samples--> Beta(alpha + s, beta + n - s)

Expectation Expected Value: (alpha / (alpha + beta))

Variance: (alpha * beta) / ((alpha + beta)^2)*(alpha+beta+1)

In [3]:
# sum([1, 1, 0, 1, 1, 0, 1])

sum([True, False, True, True])

3

In [ ]:
# Sequence Data in bool: S,S,F,S,S,F,S.
trial_data = [1, 1, 0, 1, 1, 0, 1]

class BetaDistribution():
    def calculate_expectation(alpha, beta):
        return (alpha) / (alpha + beta)

    def calculate_variance(alpha, beta):
        return (alpha * beta) / (((alpha + beta)**2) * (alpha + beta + 1))

    
    def get_posterior_distribution_params(alpha, beta, successes, trials):
        return (alpha + successes, beta + trials - successes)

    # Unnormalised
    def calculate_posterior(alpha, beta, trial_data):
        beta_params = {
            "alpha": alpha,
            "beta": beta,
            "successes": sum(trial_data),
            "trials": len(trial_data)
        }

        return BetaDistribution.get_posterior_distribution_params(**beta_params)

In [15]:
alpha = 1
beta = 1

for idx, cur_iter in enumerate(trial_data):
    cur_iter_trial = trial_data[:idx+1]
    post_alpha, post_beta = BetaDistribution.calculate_posterior(alpha, beta, cur_iter_trial)
    print(f"Posterior at Step {idx+1}: Beta({post_alpha}, {post_beta})")

print(f"Final Posterior Mean:{BetaDistribution.calculate_expectation(post_alpha, post_beta)}")

Posterior at Step 1: Beta(2, 1)
Posterior at Step 2: Beta(3, 1)
Posterior at Step 3: Beta(3, 2)
Posterior at Step 4: Beta(4, 2)
Posterior at Step 5: Beta(5, 2)
Posterior at Step 6: Beta(5, 3)
Posterior at Step 7: Beta(6, 3)
Final Posterior Mean:0.6666666666666666


Improvement:

In [22]:
import _frozen_importlib_external
from dataclasses import dataclass

@dataclass(frozen=True)
class Beta:
    alpha: float
    beta: float

    def __post_init__(self):
        if self.alpha <= 0 or self.beta <=0:
            raise ValueError("Alpha and Beta must be positive")

    @property
    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    @property
    def variance(self) -> float:
        a, b = self.alpha, self.beta
        return (a*b) / ((a + b)**2 * (a + b + 1))

    @property
    def mode(self) -> float:
        """MAP Estimate"""
        a, b = self.alpha, self.beta
        return (a - 1) / (a + b -2) if a > 1 and b > 1 else None
        
    def update(self, data: list[int]) -> "Beta":
        s, n = sum(data), len(data)
        return Beta(self.alpha + s, self.beta + n - s)

    def posterior_predictive(self) -> float:
        """P(next trial is a success. Equals the posterior mean)."""
        return self.mean

In [23]:
def trajectory(prior: Beta, data: list[int]) -> list[Beta]:
    """Posterior after each observation — item 1 of the exercise."""
    out, current = [prior], prior
    for x in data:
        current = current.update([x])
        out.append(current)
    return out

In [24]:
trajectory(Beta(1, 1), trial_data)

[Beta(alpha=1, beta=1),
 Beta(alpha=2, beta=1),
 Beta(alpha=3, beta=1),
 Beta(alpha=3, beta=2),
 Beta(alpha=4, beta=2),
 Beta(alpha=5, beta=2),
 Beta(alpha=5, beta=3),
 Beta(alpha=6, beta=3)]